# Feature Engineering — Predicting Smartphone Addiction (S6E8)

Walks through the three feature families used in the production pipeline and measures
each one's effect with a quick 3-fold LightGBM check:

1. **Target encoding** (raw value + smoothed out-of-fold mean-encoding) on every
   continuous/categorical column
2. **Ratio / sum features** between usage columns
3. **ORIG-CDF features** — placing each row against the empirical distribution of the
   small (~7500-row) real dataset the competition data was synthesized from

This notebook is illustrative and standalone: it uses a single seed and a 3-fold check
(not the tuned 5-fold/multi-seed setup used for the real leaderboard submissions), so
the AUC numbers here will differ from the numbers reported in the main README — that's
expected, not a bug.

In [ ]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score

import sys, os
sys.path.insert(0, os.getcwd())
from config import DATA

SEED = 42

train = pd.read_csv(f'{DATA}/train.csv')
test = pd.read_csv(f'{DATA}/test.csv')
orig = pd.read_csv(f'{DATA}/Smartphone_Usage_And_Addiction_Analysis_7500_Rows.csv')

y = train['addicted_label'].values
prior = y.mean()

cont_cols = ['age', 'daily_screen_time_hours', 'social_media_hours', 'gaming_hours',
             'work_study_hours', 'sleep_hours', 'notifications_per_day',
             'app_opens_per_day', 'weekend_screen_time']
cat_cols = ['gender', 'stress_level', 'academic_work_impact']
all_cats = cont_cols + cat_cols

train.shape, test.shape, y.mean()

## 1) Baseline — raw columns only

NaNs are left as-is (LightGBM handles missing values natively); this project found early
on that imputing them hurts CV, so the baseline already reflects that choice.

In [ ]:
def quick_cv_auc(X, y, n_splits=3, seed=SEED):
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    oof = np.zeros(len(y))
    for tr_idx, va_idx in skf.split(X, y):
        model = lgb.LGBMClassifier(n_estimators=400, learning_rate=0.05, num_leaves=31,
                                    random_state=seed, verbosity=-1)
        model.fit(X.iloc[tr_idx], y[tr_idx])
        oof[va_idx] = model.predict_proba(X.iloc[va_idx])[:, 1]
    return roc_auc_score(y, oof)

raw = train[all_cats].copy()
for c in all_cats:
    raw[c] = pd.to_numeric(raw[c], errors='coerce')

auc_raw = quick_cv_auc(raw, y)
print(f'Raw-only 3-fold AUC: {auc_raw:.5f}')

## 2) Target encoding

Out-of-fold smoothed mean-encoding (m-estimate smoothing) on every column, kept
*alongside* the raw value rather than replacing it — the raw+TE combination was the
single biggest early-project gain (see the main README's "Key findings").

In [ ]:
def target_encode_oof(col_train, col_test, y, n_splits=10, smooth=3.0, seed=SEED):
    prior = y.mean()
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    oof = np.zeros(len(col_train))
    for tr_idx, va_idx in skf.split(col_train, y):
        g = pd.DataFrame({'v': col_train[tr_idx], 'y': y[tr_idx]}).groupby('v')['y'].agg(['count', 'mean'])
        g['enc'] = (g['count'] * g['mean'] + smooth * prior) / (g['count'] + smooth)
        oof[va_idx] = pd.Series(col_train[va_idx]).map(g['enc'].to_dict()).fillna(prior).values
    g_full = pd.DataFrame({'v': col_train, 'y': y}).groupby('v')['y'].agg(['count', 'mean'])
    g_full['enc'] = (g_full['count'] * g_full['mean'] + smooth * prior) / (g_full['count'] + smooth)
    test_enc = pd.Series(col_test).map(g_full['enc'].to_dict()).fillna(prior).values
    return oof, test_enc

te_train = pd.DataFrame(index=train.index)
for c in all_cats:
    oof_enc, _ = target_encode_oof(train[c].astype(str).values, test[c].astype(str).values, y)
    te_train[f'{c}__te'] = oof_enc

with_te = pd.concat([raw, te_train], axis=1)
auc_te = quick_cv_auc(with_te, y)
print(f'Raw + target-encoding 3-fold AUC: {auc_te:.5f}  (delta vs raw-only: {auc_te - auc_raw:+.5f})')

## 3) Ratio / sum features

Simple derived columns between usage measures — the generator's hard decision rule
(recovered in the EDA notebook) is itself an OR/threshold rule over `daily_screen_time_hours`
and `social_media_hours`, so ratios between usage columns give the trees an easier split
to find.

In [ ]:
def add_ratio_features(df):
    out = pd.DataFrame(index=df.index)
    out['sum_components'] = df[['social_media_hours', 'gaming_hours', 'work_study_hours']].sum(axis=1, min_count=1)
    out['ratio_weekend_daily'] = df['weekend_screen_time'] / df['daily_screen_time_hours'].replace(0, np.nan)
    out['ratio_social_daily'] = df['social_media_hours'] / df['daily_screen_time_hours'].replace(0, np.nan)
    out['diff_weekend_daily'] = df['weekend_screen_time'] - df['daily_screen_time_hours']
    return out

ratio_train = add_ratio_features(train)
with_ratio = pd.concat([with_te, ratio_train], axis=1)
auc_ratio = quick_cv_auc(with_ratio, y)
print(f'+ ratio/sum features 3-fold AUC: {auc_ratio:.5f}  (delta: {auc_ratio - auc_te:+.5f})')

## 4) ORIG-CDF features

The competition data is synthetic, generated from a small (~7500-row) real dataset
(`Smartphone_Usage_And_Addiction_Analysis_7500_Rows.csv`, kept under `data/` — see the
EDA notebook for how this source was identified). These features place every train/test
row against that real dataset's *class-conditional* distributions:

- `__orig_cdf` — percentile position in the reference distribution
- `__orig_cdf_gap` — difference between the `y=0` and `y=1` conditional CDFs at that value
  (a direct separation signal)
- `__orig_q50_dist` — distance from the reference median (overall / per class)

Rows that are exact duplicates of training rows are excluded from the reference set
first, to avoid the model implicitly leaking its own label.

In [ ]:
RAW_COLS_ORIG = ['age', 'daily_screen_time_hours', 'social_media_hours', 'gaming_hours',
                  'work_study_hours', 'sleep_hours', 'notifications_per_day',
                  'app_opens_per_day', 'weekend_screen_time', 'gender', 'stress_level',
                  'academic_work_impact']
NUM_ORIG = RAW_COLS_ORIG[:9]
CDF_COLS = ['daily_screen_time_hours', 'weekend_screen_time', 'social_media_hours']


def row_hash(df, cols):
    parts = []
    for c in cols:
        if c in NUM_ORIG:
            parts.append(df[c].astype(float).round(8).fillna(-999999.0).astype(str))
        else:
            parts.append(df[c].astype(str).fillna('__MISSING__'))
    return pd.Series(list(zip(*parts))).astype(str)


train_hash = set(row_hash(train, RAW_COLS_ORIG))
orig_hash = row_hash(orig, RAW_COLS_ORIG)
orig_clean = orig.loc[~orig_hash.isin(train_hash)].drop_duplicates(subset=RAW_COLS_ORIG).reset_index(drop=True)
orig_y = orig_clean['addicted_label'].values.astype(np.int8)


def empirical_cdf(values, sorted_ref):
    values = np.asarray(values, dtype=np.float64)
    result = np.full(len(values), np.nan)
    valid = np.isfinite(values)
    if len(sorted_ref) > 0:
        result[valid] = np.searchsorted(sorted_ref, values[valid], side='right') / len(sorted_ref)
    return result


orig_cdf_train = pd.DataFrame(index=train.index)
for col in CDF_COLS:
    ref = orig_clean[col].astype(float).values
    ref_sorted = np.sort(ref[np.isfinite(ref)])
    v = train[col].astype(float).values
    orig_cdf_train[f'{col}__orig_cdf'] = empirical_cdf(v, ref_sorted)

    v0 = ref[(orig_y == 0) & np.isfinite(ref)]
    v1 = ref[(orig_y == 1) & np.isfinite(ref)]
    c0 = empirical_cdf(v, np.sort(v0))
    c1 = empirical_cdf(v, np.sort(v1))
    orig_cdf_train[f'{col}__orig_cdf_gap'] = c0 - c1

with_orig = pd.concat([with_ratio, orig_cdf_train], axis=1)
auc_orig = quick_cv_auc(with_orig, y)
print(f'+ ORIG-CDF features 3-fold AUC: {auc_orig:.5f}  (delta: {auc_orig - auc_ratio:+.5f})')

## Summary

| Stage | Features | 3-fold AUC |
|---|---|---|
| Raw only | 12 | see output above |
| + target encoding | 24 | ... |
| + ratio/sum | 28 | ... |
| + ORIG-CDF | 34 | ... |

Each stage should show a small positive delta. In the real project these deltas were
measured far more carefully (5-fold, multiple seeds, tuned hyperparameters, and — for
the smallest deltas — verified directly on the leaderboard rather than trusted from CV
alone; see the "Key findings" section of the main README for the calibration lesson that
came out of that). This notebook exists to make the *mechanism* of each feature family
inspectable, not to reproduce the exact production numbers.